# Log4j CFG Extraction with Soot (Notebook Wrapper)

This notebook runs and inspects the CFG extraction pipeline implemented in `scripts/extract_log4j_cfg.py`. The script remains the single source of truth, so notebook experiments and command-line runs produce the same artifacts.

The pipeline compiles legacy Log4j 1.0 Java source, extracts statement-level method CFGs with Soot `ExceptionalUnitGraph`, and aggregates methods into file-level graphs aligned with the PROMISE dataset labels.

## 0) Setup Paths and Imports

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

# Resolve repo root whether the notebook is run from repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_log4j_cfg.py'
CFG_OUTPUT_DIR = REPO_ROOT / 'outputs' / 'log4j' / 'cfg'

print('Repository root:', REPO_ROOT)
print('Extraction script:', SCRIPT_PATH)
print('CFG output directory:', CFG_OUTPUT_DIR)

## 1) Run CFG Extraction

The extractor reads `outputs/log4j/log4j_name_to_source_mapping.csv`, compiles the legacy Java source with ECJ, and runs the Soot backend. Soot converts each concrete method body to Jimple and builds an `ExceptionalUnitGraph`. ECJ/Soot diagnostics are saved under `outputs/log4j/cfg/*.log` instead of being printed as notebook error output.

It exports:
- readable file-level graph JSON files
- structural node feature tensors
- exact CFG node type id tensors
- edge index and edge type tensors
- stable node and edge type vocabularies
- a summary and markdown report

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print('Compiler/Soot diagnostics were captured. Character count:', len(result.stderr))

## 2) Inspect Extraction Summary

In [ ]:
summary = json.loads((CFG_OUTPUT_DIR / 'cfg_summary.json').read_text())
summary

In [ ]:
cfg_index = pd.read_csv(CFG_OUTPUT_DIR / 'cfg_index.csv')
print('Generated graphs:', len(cfg_index))
print()
print('Extraction modes:')
print(cfg_index['extraction_mode'].value_counts())
cfg_index.head()

In [ ]:
parse_failures = json.loads((CFG_OUTPUT_DIR / 'parse_failures.json').read_text())
pd.DataFrame(parse_failures)

## 3) Understand the CFG Tensors

Each CFG node is stored in two parts, and each edge is stored in two aligned tensors:

```text
node_type_id.npy -> exact CFG node type id
x.npy            -> [source_line, has_snippet, is_synthetic]
edge_index.npy   -> [source_node_ids, target_node_ids]
edge_type.npy    -> exact edge type id for each edge_index column
```

The learnable embeddings are not stored in the dataset. During GNN training, concatenate a learned node type embedding with the three extracted node features. Use edge type ids with relation-aware message passing or edge embeddings.

In [ ]:
node_type_vocab = json.loads((CFG_OUTPUT_DIR / 'node_type_vocab.json').read_text())
edge_type_vocab = json.loads((CFG_OUTPUT_DIR / 'edge_type_vocab.json').read_text())
id_to_node_type = {node_type_id: node_type for node_type, node_type_id in node_type_vocab.items()}
id_to_edge_type = {edge_type_id: edge_type for edge_type, edge_type_id in edge_type_vocab.items()}

print('CFG node type vocabulary:')
display(pd.DataFrame(sorted(node_type_vocab.items(), key=lambda item: item[1]), columns=['node_type', 'node_type_id']))
print('CFG edge type vocabulary:')
display(pd.DataFrame(sorted(edge_type_vocab.items(), key=lambda item: item[1]), columns=['edge_type', 'edge_type_id']))

## 4) Inspect One Example Graph

In [ ]:
EXAMPLE_CLASS = 'org.apache.log4j.helpers.BoundedFIFO'
tensor_dir = CFG_OUTPUT_DIR / 'tensors'

graph = json.loads((CFG_OUTPUT_DIR / 'graphs' / f'{EXAMPLE_CLASS}.json').read_text())
structural_x = np.load(tensor_dir / f'{EXAMPLE_CLASS}_x.npy')
node_type_ids = np.load(tensor_dir / f'{EXAMPLE_CLASS}_node_type_id.npy')
edge_index = np.load(tensor_dir / f'{EXAMPLE_CLASS}_edge_index.npy')
edge_type_ids = np.load(tensor_dir / f'{EXAMPLE_CLASS}_edge_type.npy')

print('Class:', EXAMPLE_CLASS)
print('methods:', len(graph['methods']))
print('structural_x shape:', structural_x.shape)
print('node_type_ids shape:', node_type_ids.shape)
print('edge_index shape:', edge_index.shape)
print('edge_type_ids shape:', edge_type_ids.shape)

In [ ]:
node_preview = pd.DataFrame({
    'node_id': np.arange(len(node_type_ids)),
    'node_type_id': node_type_ids,
    'node_type': [id_to_node_type[int(node_type_id)] for node_type_id in node_type_ids],
    'source_line': structural_x[:, 0],
    'has_snippet': structural_x[:, 1],
    'is_synthetic': structural_x[:, 2],
})
node_preview.head(20)

In [ ]:
edge_preview = pd.DataFrame({
    'source_node_id': edge_index[0],
    'target_node_id': edge_index[1],
    'edge_type_id': edge_type_ids,
    'edge_type': [id_to_edge_type[int(edge_type_id)] for edge_type_id in edge_type_ids],
})
edge_preview.head(25)

## 5) Assemble Features Inside the GNN

The extraction output is designed for a relation-aware GNN. The model learns both node type embeddings and, if the selected GNN layer supports them, edge type embeddings.

```python
import torch
from torch import nn

node_type_embedding = nn.Embedding(num_embeddings=10, embedding_dim=32)
edge_type_embedding = nn.Embedding(num_embeddings=5, embedding_dim=8)

complete_node_x = torch.cat([node_type_embedding(node_type_id), structural_x], dim=1)
edge_x = edge_type_embedding(edge_type_id)
# complete_node_x shape: [num_nodes, 35]
```

After message passing, pool node embeddings into one CFG representation per Java class. That file-level representation can later be fused with AST and NDG representations before defect classification.

## 6) Inspect the Generated Report

In [ ]:
report_text = (CFG_OUTPUT_DIR / 'cfg_report.md').read_text()

try:
    from IPython.display import Markdown, display
    display(Markdown(report_text))
except ImportError:
    print(report_text)